## Lista de experimentos a realizar
- Solo tn baseline: **0.2823**
- 100 features + autogluom baseline: **0.2574**
- Usar custom metric sin medium_quality preset: **0.7**
### a partir de aca todas con medium_quality por velocidad
- Usar custom metric con medium_quality preset y 600 budget: **0.4967** -> (no hay error, no sirve esta metrica custom para seleccion o que)
- Poner manualmente las static_features_df fijas, arreglar los types en las features  **0.2297**
- static_features_df y WAPE **0.2417**
- Escalar las features **0.2816**
- Test pesos 0 para t+1 y 1 para t+2 **0.3098**
- Escalar el target y las features: **0.3238**
- Escalar las features (sin scaling tn) con preset mas pesado: **0.38**
- TFT con regressor covariant, optimizado: **0.3013**
- Configurar feates de date como known features **0.29**
- probar con WAPE y horizon [0, 1]
- Rellenar con 0s para que llegue al largo minimo de la serie (9)
- Predecir la diferencia +1 y la diferencia +2, la prediccion final es la suma de ambas (con lightgbm no funciono pero quiza con esto si) **0.31**
- Analizar los hyperparametros de autoglom (mas que nada para tft)
- Hacer 2 modelos, uno a nivel product-cliente para los mejores 20 clientes otro para producto sumado


In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


DATA_FOLDER = "./"

#df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_best_customers_50c.pickle")
df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_grouped.pickle")
#if not "serie_id" in df.columns:
#    df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)

product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()

/home/fede/.venvs/labo3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df

,product_id,fecha,cust_request_qty,cust_request_qty_agg_std,cust_request_qty_agg_mean,cust_request_qty_agg_max,cust_request_tn,tn,tn_agg_std,tn_agg_mean,...,cust_request_qty_sku_size_vendidas_div,cust_request_qty_product_id_vendidas,cust_request_qty_product_id_vendidas_div,cust_request_qty_customer_id_vendidas,cust_request_qty_customer_id_vendidas_div,tn_customer_vendidas,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight
0,20001,2017-01,479,3.718421,1.106236,49,937.727173,934.772217,13.506243,2.158827,...,0.089801,479,1.0,187176,0.002559,34057.316406,34057.316406,1.0,934.772217,2.744703e-02
36,20002,2017-01,391,3.148312,0.903002,34,555.186523,550.157043,6.959879,1.270571,...,0.073303,391,1.0,187176,0.002089,34057.316406,34057.316406,1.0,550.157043,1.615386e-02
72,20003,2017-01,438,2.854067,1.011547,28,1067.815430,1063.458374,11.015188,2.456024,...,0.321822,438,1.0,187176,0.002340,34057.316406,34057.316406,1.0,1063.458374,3.122555e-02
108,20004,2017-01,339,2.361729,0.782910,37,569.373962,555.916138,8.193350,1.283871,...,0.105084,339,1.0,187176,0.001811,34057.316406,34057.316406,1.0,555.916138,1.632296e-02
144,20005,2017-01,249,1.754499,0.575058,28,494.600830,494.270111,7.719429,1.141501,...,0.290210,249,1.0,187176,0.001330,34057.316406,34057.316406,1.0,494.270111,1.451289e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31384,21265,2019-12,5,0.091209,0.008375,1,0.050070,0.050070,0.001378,0.000084,...,0.454545,5,1.0,123878,0.000040,26217.068359,26217.068359,1.0,0.050070,1.909825e-06
31394,21266,2019-12,6,0.115419,0.010050,2,0.051210,0.051210,0.001381,0.000086,...,0.545455,6,1.0,123878,0.000048,26217.068359,26217.068359,1.0,0.051210,1.953308e-06
31404,21267,2019-12,4,0.081648,0.006700,1,0.015690,0.015690,0.000447,0.000026,...,0.002221,4,1.0,123878,0.000032,26217.068359,26217.068359,1.0,0.015690,5.984650e-07
31449,21271,2019-12,4,0.100111,0.006700,2,0.002980,0.002980,0.000071,0.000005,...,0.057971,4,1.0,123878,0.000032,26217.068359,26217.068359,1.0,0.002980,1.136664e-07


In [12]:
import numpy as np
numeric_df = df.select_dtypes(include=[np.number])
total_infs = np.isinf(numeric_df.values).sum()
print(f"Total infs: {total_infs}")
df[numeric_df.columns] = df[numeric_df.columns].replace([np.inf, -np.inf], np.nan)

Total infs: 0


In [13]:
df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M'))  # último día del mes
df

,product_id,fecha,cust_request_qty,cust_request_qty_agg_std,cust_request_qty_agg_mean,cust_request_qty_agg_max,cust_request_tn,tn,tn_agg_std,tn_agg_mean,...,cust_request_qty_sku_size_vendidas_div,cust_request_qty_product_id_vendidas,cust_request_qty_product_id_vendidas_div,cust_request_qty_customer_id_vendidas,cust_request_qty_customer_id_vendidas_div,tn_customer_vendidas,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight
0,20001,2017-01-31,479,3.718421,1.106236,49,937.727173,934.772217,13.506243,2.158827,...,0.089801,479,1.0,187176,0.002559,34057.316406,34057.316406,1.0,934.772217,2.744703e-02
36,20002,2017-01-31,391,3.148312,0.903002,34,555.186523,550.157043,6.959879,1.270571,...,0.073303,391,1.0,187176,0.002089,34057.316406,34057.316406,1.0,550.157043,1.615386e-02
72,20003,2017-01-31,438,2.854067,1.011547,28,1067.815430,1063.458374,11.015188,2.456024,...,0.321822,438,1.0,187176,0.002340,34057.316406,34057.316406,1.0,1063.458374,3.122555e-02
108,20004,2017-01-31,339,2.361729,0.782910,37,569.373962,555.916138,8.193350,1.283871,...,0.105084,339,1.0,187176,0.001811,34057.316406,34057.316406,1.0,555.916138,1.632296e-02
144,20005,2017-01-31,249,1.754499,0.575058,28,494.600830,494.270111,7.719429,1.141501,...,0.290210,249,1.0,187176,0.001330,34057.316406,34057.316406,1.0,494.270111,1.451289e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31384,21265,2019-12-31,5,0.091209,0.008375,1,0.050070,0.050070,0.001378,0.000084,...,0.454545,5,1.0,123878,0.000040,26217.068359,26217.068359,1.0,0.050070,1.909825e-06
31394,21266,2019-12-31,6,0.115419,0.010050,2,0.051210,0.051210,0.001381,0.000086,...,0.545455,6,1.0,123878,0.000048,26217.068359,26217.068359,1.0,0.051210,1.953308e-06
31404,21267,2019-12-31,4,0.081648,0.006700,1,0.015690,0.015690,0.000447,0.000026,...,0.002221,4,1.0,123878,0.000032,26217.068359,26217.068359,1.0,0.015690,5.984650e-07
31449,21271,2019-12-31,4,0.100111,0.006700,2,0.002980,0.002980,0.000071,0.000005,...,0.057971,4,1.0,123878,0.000032,26217.068359,26217.068359,1.0,0.002980,1.136664e-07


In [14]:
TEST_DATE = 33

def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index
df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)

test_index, train_index, train_scaler_index = get_indexes(df)
train_df = df.loc[train_index].copy()
test_df = df.loc[test_index].copy()

/tmp/ipykernel_104391/2641507396.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)


In [15]:
train_data = TimeSeriesDataFrame.from_data_frame(
    train_df[["fecha", "product_id", "tn"]],
    id_column="product_id",
    timestamp_column="fecha",
)
train_data.head()

,,tn
item_id,timestamp,
20001,2017-01-31,934.772217
20002,2017-01-31,550.157043
20003,2017-01-31,1063.458374
20004,2017-01-31,555.916138
20005,2017-01-31,494.270111


In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    #eval_metric=, 
)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)

No path specified. Models will be saved in: "AutogluonModels/ag-20250627_232217"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_232217'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.99 GB / 15.32 GB (39.1%)
Disk Space Avail:   106.33 GB / 575.67 GB (18.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection':

In [ ]:
test_pred = predictor.predict(train_data)
test_pred

data with frequency 'IRREG' has been resampled to frequency 'ME'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean          0.1          0.2          0.3  \
item_id timestamp                                                        
20001   2019-11-30  1579.858092  1165.677723  1305.227425  1407.286870   
        2019-12-31  1524.584403  1093.464330  1236.870837  1343.081829   
20002   2019-11-30  1363.338863   957.385227  1093.176932  1192.724619   
        2019-12-31  1232.476895   816.163986   953.461207  1055.504048   
20003   2019-11-30  1069.190355   772.197739   869.892310   943.182591   
...                         ...          ...          ...          ...   
21239   2019-12-31     0.184780     0.184779     0.184780     0.184780   
21242   2019-11-30     0.295850     0.295849     0.295850     0.295850   
        2019-12-31     0.295850     0.295849     0.295850     0.295850   
21247   2019-11-30     0.188710     0.188709     0.188710     0.188710   
        2019-12-31     0.188710     0.188709     0.188710     0.188710   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2019-11-30  1496.808481  1580.264077  1665.497453  1758.916030   
        2019-12-31  1435.829361  1523.335349  1612.659394  1711.649609   
20002   2019-11-30  1280.703253  1363.702372  1450.177597  1549.479082   
        2019-12-31  1145.371134  1231.343423  1320.912178  1424.413455   
20003   2019-11-30  1007.843541  1069.545115  1134.863445  1208.754592   
...                         ...          ...          ...          ...   
21239   2019-12-31     0.184780     0.184780     0.184780     0.184780   
21242   2019-11-30     0.295850     0.295850     0.295850     0.295850   
        2019-12-31     0.295850     0.295850     0.295850     0.295850   
21247   2019-11-30     0.188710     0.188710     0.188710     0.188710   
        2019-12-31     0.188710     0.188710     0.188710     0.188710   

                            0.8          0.9  
item_id timestamp                             
20001   2019-11-30  1872.639866  2046.046401  
        2019-12-31  1832.135753  2019.416939  
20002   2019-11-30  1675.885845  1879.992234  
        2019-12-31  1557.238957  1774.350136  
20003   2019-11-30  1300.811869  1447.290784  
...                         ...          ...  
21239   2019-12-31     0.184781     0.184783  
21242   2019-11-30     0.295850     0.295852  
        2019-12-31     0.295851     0.295853  
21247   2019-11-30     0.188710     0.188712  
        2019-12-31     0.188711     0.188713  

[2458 rows x 10 columns]

In [ ]:
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})
predictions = predictions[predictions["product_id"].isin(product_ids)]

# le agrego la columna target de test_df mergeando por product_id
#test_df = test_df[['product_id', 'fecha', 'target']].drop_duplicates()
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
predictions

,product_id,mean,target,abs_error
0,20001,1524.584403,1504.688599,19.895804
1,20002,1232.476895,1087.308594,145.168301
2,20003,944.637809,892.501282,52.136527
3,20004,785.307105,637.900024,147.407080
4,20005,745.976717,593.244446,152.732271
...,...,...,...,...
775,21263,0.023225,0.012700,0.010525
776,21265,0.086050,0.050070,0.035980
777,21266,0.091993,0.051210,0.040783
778,21267,0.076816,0.015690,0.061126


In [ ]:
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")

Total Absolute Error: 0.2823


### benchmark -  0.2823

In [ ]:
# ahora agrego features:

train_data = TimeSeriesDataFrame.from_data_frame(
    train_df.drop(columns=['target']),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
)
train_data.head()


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    #eval_metric=total_abs_error_scorer, 
)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)

No path specified. Models will be saved in: "AutogluonModels/ag-20250627_232541"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_232541'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.40 GB / 15.32 GB (35.2%)
Disk Space Avail:   106.31 GB / 575.67 GB (18.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection':

train_data with frequency 'IRREG' has been resampled to frequency 'ME'.
Provided train_data has 29696 rows, 1229 time series. Median time series length is 31 (min=1, max=34). 
	Removing 163 short time series from train_data. Only series with length >= 7 will be used for training.
	After filtering, train_data has 29183 rows, 1066 time series. Median time series length is 34 (min=7, max=34). 

Provided data contains following columns:
	target: 'tn'
	past_covariates:
		categorical:        ['cat1', 'cat2', 'cat3', 'brand']
		continuous (float): ['cust_request_qty', 'cust_request_qty_agg_std', 'cust_request_qty_agg_mean', 'cust_request_qty_agg_max', 'cust_request_tn', 'tn_agg_std', ...]

AutoGluon will ignore following non-numeric/non-informative columns:
	ignored covariates:      ['cust_request_qty_product_id_vendidas', 'cust_request_qty_product_id_vendidas_div', 'tn_customer_vendidas', 'tn_customer_weight', 'tn_product_vendidas', 'tn_product_weight', 'tn_total_vendidas']

To learn how to 

In [ ]:
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})
predictions = predictions[predictions["product_id"].isin(product_ids)]

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions

data with frequency 'IRREG' has been resampled to frequency 'ME'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Total Absolute Error: 0.2574


,product_id,mean,target,abs_error
0,20001,1439.440224,1504.688599,65.248375
1,20002,1164.850472,1087.308594,77.541878
2,20003,937.923775,892.501282,45.422493
3,20004,746.564568,637.900024,108.664543
4,20005,711.350978,593.244446,118.106533
...,...,...,...,...
775,21263,0.008034,0.012700,0.004666
776,21265,0.086070,0.050070,0.036000
777,21266,0.092753,0.051210,0.041543
778,21267,0.077849,0.015690,0.062159


## Total Absolute Error: 0.2574

In [ ]:
test_pred = predictor.predict(train_data)
test_pred

data with frequency 'IRREG' has been resampled to frequency 'ME'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean          0.1          0.2          0.3  \
item_id timestamp                                                        
20001   2019-11-30  1503.678198  1118.437846  1253.366321  1349.326790   
        2019-12-31  1439.440224  1040.895343  1178.600154  1276.669015   
20002   2019-11-30  1290.472968   909.723194  1042.055237  1135.573962   
        2019-12-31  1164.850472   775.698798   909.670234  1004.063041   
20003   2019-11-30  1064.415923   778.282491   876.128774   948.637389   
...                         ...          ...          ...          ...   
21239   2019-12-31     0.184200     0.183465     0.183933     0.183896   
21242   2019-11-30     0.295570     0.294761     0.295248     0.295152   
        2019-12-31     0.295280     0.294517     0.294986     0.294963   
21247   2019-11-30     0.188418     0.187637     0.188121     0.188010   
        2019-12-31     0.188129     0.187397     0.187863     0.187826   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2019-11-30  1424.155942  1503.678198  1581.264017  1668.754786   
        2019-12-31  1355.572584  1439.440224  1516.652736  1608.388820   
20002   2019-11-30  1208.969604  1290.472968  1377.602691  1475.449738   
        2019-12-31  1079.795495  1164.850472  1250.859729  1352.199202   
20003   2019-11-30   997.601072  1064.415923  1126.390992  1198.558553   
...                         ...          ...          ...          ...   
21239   2019-12-31     0.183992     0.184200     0.184442     0.184623   
21242   2019-11-30     0.295313     0.295570     0.295776     0.295984   
        2019-12-31     0.295060     0.295280     0.295526     0.295721   
21247   2019-11-30     0.188172     0.188418     0.188618     0.188811   
        2019-12-31     0.187923     0.188129     0.188370     0.188549   

                            0.8          0.9  
item_id timestamp                             
20001   2019-11-30  1775.902009  1935.910068  
        2019-12-31  1721.893129  1890.565172  
20002   2019-11-30  1591.815979  1780.939500  
        2019-12-31  1476.376376  1672.519159  
20003   2019-11-30  1281.647139  1422.306061  
...                         ...          ...  
21239   2019-12-31     0.184759     0.184908  
21242   2019-11-30     0.296177     0.296358  
        2019-12-31     0.295860     0.296017  
21247   2019-11-30     0.189004     0.189176  
        2019-12-31     0.188687     0.188834  

[2458 rows x 10 columns]

In [ ]:
test_pred = predictor.predict(test_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions

data with frequency 'IRREG' has been resampled to frequency 'ME'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Total Absolute Error: 0.4967


,product_id,mean,target,abs_error
0,20001,1561.506499,1504.688599,56.817901
1,20002,1979.538061,1087.308594,892.229467
2,20003,1081.367792,892.501282,188.866510
3,20004,1064.697982,637.900024,426.797958
4,20005,996.784420,593.244446,403.539974
...,...,...,...,...
967,21266,0.119737,0.051210,0.068527
968,21267,0.098058,0.015690,0.082368
969,21269,0.039051,0.000000,0.039051
970,21271,0.025219,0.002980,0.022239


In [ ]:
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'serie_id'})
# separo serie_id en 2 columnas int16: product_id y customer_id 
predictions[['product_id', 'customer_id']] = predictions['serie_id'].str.split('_', expand=True)
predictions['product_id'] = predictions['product_id'].astype('int16')
predictions['customer_id'] = predictions['customer_id'].astype('int16')
predictions = predictions.merge(test_df[['product_id', "customer_id", 'target']].drop_duplicates(), on=['product_id', "customer_id"], how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")
predictions



Total Absolute Error: 0.2404


,mean,target,abs_error
product_id,,,
20001,1164.814862,1504.688599,339.873737
20002,904.947649,1087.308594,182.360945
20003,703.130003,892.501282,189.371279
20004,555.600654,637.900024,82.299370
20005,562.770990,593.244446,30.473456
...,...,...,...
21263,0.021699,0.012700,0.008999
21265,0.062746,0.050070,0.012676
21266,0.068692,0.051210,0.017482


In [16]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns


static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])


static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()



predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)

In [19]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns


static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])


static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)


predictor.fit(
    train_data,
    num_val_windows=2
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']], on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / predictions['target'].sum()
print(f"Total Absolute Error: {total_error:.4f}")
predictions

Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250702_003824'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       10.61 GB / 15.32 GB (69.3%)
Disk Space Avail:   67.60 GB / 575.67 GB (11.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled to frequency 'ME'.
Provided train_da

KeyboardInterrupt: 

In [ ]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns


static_features_df = pd.DataFrame({
    'cat1': df.groupby('serie_id')['cat1'].first(),
    'cat2': df.groupby('serie_id')['cat2'].first(),
    'cat3': df.groupby('serie_id')['cat3'].first(),
    'brand': df.groupby('serie_id')['brand'].first(),
    'sku_size': df.groupby('serie_id')['sku_size'].first(),
    "product_id": df.groupby('serie_id')['product_id'].first(),
    "customer_id": df.groupby('serie_id')['customer_id'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="serie_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)


predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions


No path specified. Models will be saved in: "AutogluonModels/ag-20250629_032135"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250629_032135'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       7.51 GB / 15.32 GB (49.0%)
Disk Space Avail:   91.79 GB / 575.67 GB (15.9%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': 

KeyboardInterrupt: 

In [ ]:

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions

Total Absolute Error: 0.2220


,mean,target,abs_error
product_id,,,
20001,1353.335991,1504.688599,151.352608
20002,1091.860282,1087.308594,4.551688
20003,799.862107,892.501282,92.639174
20004,555.017988,634.276794,79.258806
20005,579.635181,593.244446,13.609264
...,...,...,...
21263,-0.005763,0.012700,0.018463
21265,0.011703,0.050070,0.038367
21266,0.012220,0.051210,0.038990


0.2220 con dataset de best customer + resto

## 0.2297

In [ ]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target']), 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    eval_metric="WAPE"
)


predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})
predictions = predictions[predictions["product_id"].isin(product_ids)]

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions



No path specified. Models will be saved in: "AutogluonModels/ag-20250627_234803"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_234803'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.63 GB / 15.32 GB (36.8%)
Disk Space Avail:   106.21 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection'

Total Absolute Error: 0.2417


,product_id,mean,target,abs_error
0,20001,1460.726728,1504.688599,43.961871
1,20002,1173.580340,1087.308594,86.271746
2,20003,912.827274,892.501282,20.325992
3,20004,756.074723,637.900024,118.174699
4,20005,729.695233,593.244446,136.450787
...,...,...,...,...
775,21263,0.003773,0.012700,0.008927
776,21265,0.078674,0.050070,0.028604
777,21266,0.083098,0.051210,0.031888
778,21267,0.076368,0.015690,0.060678


In [ ]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
import re


static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
    ]
    + [r"cust_request_qty_diff_*"],
}

def scale_df(df, transformations, train_scaler_index):
    numeric_columns = df.select_dtypes(include=["float64", "float32", "int32", "int64"]).columns
    print(numeric_columns)
    df_scaled = df  # no hago copy intencionalmente
    train_scaler_df = df_scaled.loc[train_scaler_index]


    prod_stats = train_scaler_df.groupby("product_id")[
        list(transformations.keys())
    ].agg(["mean", "std"])
    #print(prod_stats.head())

    def custom_group_stats(group):
        product_id = group.name[1]
        row = {"customer_id": group.name[0], "product_id": product_id}
        for col in transformations.keys():
            std_prod = prod_stats.loc[product_id, (col, "std")]
            mean = group[col].mean()
            std = std_prod
            row[f"{col}_mean"] = mean
            row[f"{col}_std"] = std
        return pd.Series(row)

    group_stats = (
        train_scaler_df.groupby(["customer_id", "product_id"])
        .apply(custom_group_stats)
        .reset_index(drop=True)
    )
    print(group_stats.head())

    # Mergear las stats al df original
    df_scaled = df_scaled.merge(
        group_stats, on=["product_id", "customer_id"], how="left"
    )
    df_scaled = df_scaled.set_index(df.index)

    scaled_cols = {}
    for trainer, regex_cols in transformations.items():
        for col in regex_cols:
            # Usar regex para seleccionar las columnas que coinciden
            # chequear si la columna es un regex
            matching_cols = [c for c in numeric_columns if re.match(col, c)]
            if not matching_cols:
                continue  # Si no hay columnas que coincidan, saltar

            # Calcular la media y desviación estándar para cada
            print(f"Processing trainer: {trainer} with columns: {matching_cols}")
            # Escalar las columnas
            for col in matching_cols:
                scaled_cols[col + "_scaled"] = (df_scaled[col]) / df_scaled[
                    trainer + "_std"
                ]

    # Crear un DataFrame con todas las columnas escaladas
    scaled_df = pd.DataFrame(scaled_cols, index=df_scaled.index)

    # Concatenar de una sola vez
    df_scaled = pd.concat([df_scaled, scaled_df], axis=1)
    aux_cols = [col + "_mean" for col in list(transformations.keys())] + [
        col + "_std" for col in list(transformations.keys())
    ]
    df_scaled = df_scaled.drop(columns=aux_cols)
    return df_scaled, group_stats

train_df_no_static, group_stats = scale_df(train_df_no_static, transformations, train_df_no_static.index)


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target']), 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})
predictions = predictions[predictions["product_id"].isin(product_ids)]

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions


Index(['product_id', 'cust_request_qty', 'cust_request_qty_agg_std',
       'cust_request_qty_agg_mean', 'cust_request_qty_agg_max',
       'cust_request_tn', 'tn', 'tn_agg_std', 'tn_agg_mean', 'tn_agg_max',
       ...
       'cust_request_qty_product_id_vendidas',
       'cust_request_qty_product_id_vendidas_div',
       'cust_request_qty_customer_id_vendidas',
       'cust_request_qty_customer_id_vendidas_div', 'tn_customer_vendidas',
       'tn_total_vendidas', 'tn_customer_weight', 'tn_product_vendidas',
       'tn_product_weight', 'target'],
      dtype='object', length=101)


/tmp/ipykernel_31984/347074873.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(custom_group_stats)
No path specified. Models will be saved in: "AutogluonModels/ag-20250627_234206"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_234206'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.68 GB / 15.32 GB (37.1%)
Disk Space Avail:   106.25 GB / 575.6

   customer_id  product_id      tn_mean      tn_std  cust_request_qty_mean  \
0          1.0     20001.0  1395.244995  306.472504             438.500000   
1          1.0     20002.0   994.893127  303.828033             423.970588   
2          1.0     20003.0   887.157593  300.572388             405.029412   
3          1.0     20004.0   671.067993  230.896774             412.735294   
4          1.0     20005.0   646.795959  224.505920             348.441176   

   cust_request_qty_std  
0             96.011442  
1             86.972126  
2             82.056331  
3             74.567565  
4             78.232440  
Processing trainer: tn with columns: ['tn']
Processing trainer: tn with columns: ['tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_11', 'tn_lag_15']
Processing trainer: tn with columns: ['tn_rolling_mean_6', 'tn_rolling_mean_12', 'tn_rolling_mean_12_lag_1', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_12_lag_11', 'tn_rolling_mean_12_lag_15', 'tn_roll

train_data with frequency 'IRREG' has been resampled to frequency 'ME'.
Provided train_data has 29696 rows, 1229 time series. Median time series length is 31 (min=1, max=34). 
	Removing 163 short time series from train_data. Only series with length >= 7 will be used for training.
	After filtering, train_data has 29183 rows, 1066 time series. Median time series length is 34 (min=7, max=34). 

Provided data contains following columns:
	target: 'tn'
	past_covariates:
		categorical:        []
		continuous (float): ['cust_request_qty', 'cust_request_qty_agg_std', 'cust_request_qty_agg_mean', 'cust_request_qty_agg_max', 'cust_request_tn', 'tn_agg_std', ...]
	static_features:
		categorical:        ['cat1', 'cat2', 'cat3', 'brand', 'sku_size']
		continuous (float): []

AutoGluon will ignore following non-numeric/non-informative columns:
	ignored covariates:      ['cust_request_qty_product_id_vendidas', 'cust_request_qty_product_id_vendidas_div', 'cust_request_qty_product_id_vendidas_scaled', '

Total Absolute Error: 0.2816


,product_id,mean,target,abs_error
0,20001,1522.548955,1504.688599,17.860356
1,20002,1234.124855,1087.308594,146.816261
2,20003,942.819764,892.501282,50.318482
3,20004,783.893124,637.900024,145.993100
4,20005,745.106718,593.244446,151.862272
...,...,...,...,...
775,21263,0.021351,0.012700,0.008651
776,21265,0.086407,0.050070,0.036337
777,21266,0.092393,0.051210,0.041183
778,21267,0.075741,0.015690,0.060051


In [ ]:
known_covariate_names = [
    'year',
    'mes',
    'quarter',
    'date_id',
]


static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df,
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    known_covariates_names=known_covariate_names

)

# al dataset futuro le agrego las columnas de covariables conocidas
future_data = predictor.make_future_data_frame(train_data)
future_data["year"] = future_data["timestamp"].dt.year
future_data["mes"] = future_data["timestamp"].dt.month
future_data["quarter"] = future_data["timestamp"].dt.quarter
# date id es el last_date id de train +1 y +2 para cada fecha
last_date_id = test_df['date_id'].max()
# creo un date_id para el futuro (1 para la primer fecha 2 para la segunda) y le sumo el ultimo date_id de test_df
num_dates = future_data['timestamp'].unique()
future_data["date_id"] =0
for i, date in enumerate(num_dates):
    # le sumo el ultimo date_id de test_df
    future_data.loc[future_data['timestamp'] == date, 'date_id'] = last_date_id + i + 1


predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data, known_covariates=future_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")


No path specified. Models will be saved in: "AutogluonModels/ag-20250628_013600"
data with frequency 'IRREG' has been resampled to frequency 'ME'.
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_013600'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.35 GB / 15.32 GB (34.9%)
Disk Space Avail:   105.89 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': ['year', 'mes', 'quarter', 'date_id'],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0

Total Absolute Error: 0.2840


In [ ]:
predictions

,product_id,mean,target,abs_error
0,20001,1527.327695,1504.688599,22.639097
1,20002,1247.586468,1087.308594,160.277874
2,20003,947.532531,892.501282,55.031249
3,20004,792.253973,637.900024,154.353949
4,20005,755.916655,593.244446,162.672209
...,...,...,...,...
1193,21263,0.020485,0.012700,0.007785
1195,21265,0.083276,0.050070,0.033206
1196,21266,0.085656,0.051210,0.034446
1197,21267,0.074853,0.015690,0.059163


In [ ]:
# el target es tn_diff_1, luego de predecir sumo las predicciones y lo sumo a tn

train_df.drop(columns=['target'], inplace=True, errors="ignore") 
static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"), 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df,
)
train_data.head()

from autogluon.common.utils.log_utils import set_logger_verbosity


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn_diff_1",
    freq="ME",

)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').sum()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target', "tn"]].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["mean"] = predictions["mean"] + predictions["tn"]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")

No path specified. Models will be saved in: "AutogluonModels/ag-20250628_011114"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_011114'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.50 GB / 15.32 GB (35.9%)
Disk Space Avail:   106.02 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection':

Total Absolute Error: 0.3282


In [ ]:
predictions

,product_id,mean,target,tn,abs_error
0,20001,1648.986447,1504.688599,1561.505493,144.297849
1,20002,1869.574190,1087.308594,1979.536377,782.265596
2,20003,929.655975,892.501282,1081.366455,37.154693
3,20004,982.086986,637.900024,1064.696289,344.186962
4,20005,894.657642,593.244446,996.782776,301.413196
...,...,...,...,...,...
1193,21263,0.000000,0.012700,0.015520,0.012700
1195,21265,0.219852,0.050070,0.109210,0.169782
1196,21266,0.228370,0.051210,0.118310,0.177160
1197,21267,0.114378,0.015690,0.096760,0.098688


In [ ]:
# el target es tn_diff_2, luego de predecir sumo las predicciones y lo sumo a tn

train_df.drop(columns=['target'], inplace=True, errors="ignore") 
static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"), 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df,
)
train_data.head()

from autogluon.common.utils.log_utils import set_logger_verbosity


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn_diff_2",
    freq="ME",

)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target', "tn"]].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["mean"] = predictions["mean"] + predictions["tn"]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_011440'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.27 GB / 15.32 GB (34.4%)
Disk Space Avail:   105.99 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn_diff_2',
 'time_limit': 600,
 'verbosity': 2}

train_data 

Total Absolute Error: 0.2928


In [ ]:
df[df["product_id"] == 20002][["date_id","tn"]]

,date_id,tn
36,0,550.157043
37,1,505.886322
38,2,834.735229
39,3,522.353638
40,4,843.437866
41,5,968.157593
42,6,845.393188
43,7,619.710754
44,8,1065.345337
45,9,857.452698


In [ ]:
# submission base
def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index

test_index_pred_final, train_index_pred_final, _ = get_indexes(df, 35)
train_df_pred_final = df.loc[train_index_pred_final].copy()
test_df_pred_final = df.loc[test_index_pred_final].copy()


# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df_pred_final.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)


predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df_pred_final[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions


No path specified. Models will be saved in: "AutogluonModels/ag-20250628_012525"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_012525'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       4.94 GB / 15.32 GB (32.3%)
Disk Space Avail:   105.94 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection':

Total Absolute Error: 0.0000


,product_id,mean,target,abs_error
0,20001,1331.910208,NaN,NaN
1,20002,1031.235790,NaN,NaN
2,20003,724.194751,NaN,NaN
3,20004,512.378919,NaN,NaN
4,20005,548.232921,NaN,NaN
...,...,...,...,...
1197,21263,-0.003709,NaN,NaN
1199,21265,0.092505,NaN,NaN
1200,21266,0.097221,NaN,NaN
1201,21267,0.031340,NaN,NaN


In [ ]:
submission = predictions[['product_id', 'mean']].rename(columns={'mean': 'tn'})
submission['tn'] = submission['tn'].clip(lower=0)  # Asegurar que las predicciones no sean negativas
submission.to_csv("submission_tft_prueba.csv", index=False)
submission

,product_id,tn
0,20001,1331.910208
1,20002,1031.235790
2,20003,724.194751
3,20004,512.378919
4,20005,548.232921
...,...,...
1197,21263,0.000000
1199,21265,0.092505
1200,21266,0.097221
1201,21267,0.031340


In [ ]:
def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index

test_index_pred_final, train_index_pred_final, _ = get_indexes(df, 35)
train_df_pred_final = df.loc[train_index_pred_final].copy()
test_df_pred_final = df.loc[test_index_pred_final].copy()

print(f"Train shape: {train_df_pred_final.shape}, Test shape: {test_df_pred_final.shape}")
print(f"DF shape: {df.shape}")

static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df_pred_final.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static, 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df,
)
train_data.head()

predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn_diff_1",
    freq="ME",

)

predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').sum()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df_pred_final[['product_id', 'target', "tn"]].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["mean"] = predictions["mean"] + predictions["tn"]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")

No path specified. Models will be saved in: "AutogluonModels/ag-20250628_003555"
Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_003555'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.07 GB / 15.32 GB (33.1%)
Disk Space Avail:   106.08 GB / 575.67 GB (18.4%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection':

Train shape: (31645, 107), Test shape: (976, 107)
DF shape: (31645, 107)


train_data with frequency 'IRREG' has been resampled to frequency 'ME'.
Provided train_data has 31645 rows (NaN fraction=3.9%), 1233 time series. Median time series length is 31 (min=1, max=36). 
	Removing 131 short time series from train_data. Only series with length >= 7 will be used for training.
	After filtering, train_data has 31147 rows (NaN fraction=3.5%), 1102 time series. Median time series length is 36 (min=7, max=36). 

Provided data contains following columns:
	target: 'tn_diff_1'
	past_covariates:
		categorical:        []
		continuous (float): ['cust_request_qty', 'cust_request_qty_agg_std', 'cust_request_qty_agg_mean', 'cust_request_qty_agg_max', 'cust_request_tn', 'tn', ...]
	static_features:
		categorical:        ['cat1', 'cat2', 'cat3', 'brand', 'sku_size']
		continuous (float): []

AutoGluon will ignore following non-numeric/non-informative columns:
	ignored covariates:      ['cust_request_qty_product_id_vendidas', 'cust_request_qty_product_id_vendidas_div', 'tn_custo

Total Absolute Error: 0.0000


In [ ]:
test_pred

mean         0.1         0.2         0.3  \
item_id timestamp                                                    
20001   2020-01-31  -89.934715 -177.745682 -138.950317 -123.845078   
        2020-02-29 -142.078918 -248.729248 -203.125015 -179.534836   
20002   2020-01-31   -1.712583  -74.591675  -46.548607  -29.520294   
        2020-02-29   16.323870  -60.081406  -32.478947  -10.816510   
20003   2020-01-31  -89.031334 -142.502945 -113.384819 -102.798599   
...                        ...         ...         ...         ...   
20770   2020-02-29    0.005262    0.003030    0.003971    0.004176   
20792   2020-01-31   -0.000344   -0.002275   -0.001614   -0.001533   
        2020-02-29    0.005026    0.002859    0.003741    0.003862   
20854   2020-01-31    0.000163   -0.001478   -0.001036   -0.000941   
        2020-02-29    0.005125    0.003324    0.004028    0.003986   

                           0.4         0.5         0.6        0.7        0.8  \
item_id timestamp                                                              
20001   2020-01-31 -106.819771  -89.934715  -66.128815 -33.151051 -11.437084   
        2020-02-29 -163.086395 -142.078918 -112.428528 -86.607086 -66.607117   
20002   2020-01-31  -23.110764   -1.712583   18.429134  44.232792  59.406086   
        2020-02-29   -1.669462   16.323870   42.170158  64.947746  79.532494   
20003   2020-01-31  -97.713318  -89.031334  -65.226372 -49.510532 -34.945797   
...                        ...         ...         ...        ...        ...   
20770   2020-02-29    0.004838    0.005262    0.005758   0.006403   0.007316   
20792   2020-01-31   -0.000831   -0.000344    0.000439   0.000822   0.001652   
        2020-02-29    0.004610    0.005026    0.005560   0.006228   0.007101   
20854   2020-01-31   -0.000222    0.000163    0.000898   0.001314   0.002073   
        2020-02-29    0.004641    0.005125    0.005519   0.006224   0.007044   

                           0.9  
item_id timestamp               
20001   2020-01-31   12.584211  
        2020-02-29  -28.681015  
20002   2020-01-31   81.897095  
        2020-02-29  110.492088  
20003   2020-01-31  -17.320396  
...                        ...  
20770   2020-02-29    0.008498  
20792   2020-01-31    0.002535  
        2020-02-29    0.008301  
20854   2020-01-31    0.002916  
        2020-02-29    0.008231  

[2466 rows x 10 columns]

In [ ]:
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').sum()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df_pred_final[['product_id', 'target', "tn"]].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["pred"] = predictions["mean"] + predictions["tn"]
predictions["pred"] = predictions["pred"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['pred'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['pred']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions

Total Absolute Error: 0.0000


,product_id,mean,target,tn,pred,abs_error
0,20001,-232.013641,NaN,1504.688599,1272.674927,NaN
1,20002,14.611287,NaN,1087.308594,1101.919922,NaN
2,20003,-101.813377,NaN,892.501282,790.687927,NaN
3,20004,-43.538685,NaN,637.900024,594.361328,NaN
4,20005,-38.060295,NaN,593.244446,555.184143,NaN
...,...,...,...,...,...,...
1197,21263,0.076987,NaN,0.012700,0.089687,NaN
1199,21265,0.111764,NaN,0.050070,0.161834,NaN
1200,21266,0.110720,NaN,0.051210,0.161930,NaN
1201,21267,0.051151,NaN,0.015690,0.066841,NaN


In [ ]:
submission = predictions[['product_id', 'mean']].rename(columns={'mean': 'tn'})
submission['tn'] = submission['tn'].clip(lower=0)  # Asegurar que las predicciones no sean negativas
submission.to_csv("submission_tft_recursivo.csv_sin_error_fatal", index=False)
submission

,product_id,tn
0,20001,1272.674927
1,20002,1101.919922
2,20003,790.687927
3,20004,594.361328
4,20005,555.184143
...,...,...
1197,21263,0.089687
1199,21265,0.161834
1200,21266,0.161930
1201,21267,0.066841


In [ ]:
def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index

if not "serie_id" in df:
    df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
    
test_index_pred_final, train_index_pred_final, _ = get_indexes(df, 35)
train_df = df.loc[train_index_pred_final].copy()
test_df = df.loc[test_index_pred_final].copy()

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"DF shape: {df.shape}")
# creo columna serie_id que es la mezcla de product_id y customer_id

# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
static_features_df = pd.DataFrame({
    'cat1': df.groupby('serie_id')['cat1'].first(),
    'cat2': df.groupby('serie_id')['cat2'].first(),
    'cat3': df.groupby('serie_id')['cat3'].first(),
    'brand': df.groupby('serie_id')['brand'].first(),
    'sku_size': df.groupby('serie_id')['sku_size'].first(),
    "product_id": df.groupby('serie_id')['product_id'].first(),
    "customer_id": df.groupby('serie_id')['customer_id'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="serie_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)

from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)


predictor.fit(
    train_data,
)
test_pred = predictor.predict(train_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
submission = predictions[['product_id', 'mean']].rename(columns={'mean': 'tn'})
submission['tn'] = submission['tn'].clip(lower=0)  # Asegurar que las predicciones no sean negativas
submission.to_csv("submission_autogluon.csv", index=False)
submission



Train shape: (664545, 119), Test shape: (20496, 119)
DF shape: (664545, 119)


No path specified. Models will be saved in: "AutogluonModels/ag-20250629_211416"
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250629_211416'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       3.38 GB / 15.32 GB (22.0%)
Disk Space Avail:   85.48 GB / 575.67 GB (14.8%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_dat

ValueError: You are trying to merge on object and int16 columns for key 'product_id'. If you wish to proceed you should use pd.concat

In [ ]:
test_df

,product_id,fecha,customer_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,cust_request_qty_product_id_vendidas_div,cust_request_qty_customer_id_vendidas,cust_request_qty_customer_id_vendidas_div,tn_customer_vendidas,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight,serie_id,target
644049,20001,2019-12-31,0,NaN,234,364.751648,358.836945,69.755478,HC,ROPA LAVADO,...,0.574939,72882,0.003211,10786.160156,26217.066406,0.411418,1504.688599,5.739348e-02,20001_0,NaN
644050,20001,2019-12-31,10001,0.0,18,214.721848,180.219376,69.755478,HC,ROPA LAVADO,...,0.044226,3121,0.005767,2374.469238,26217.066406,0.090570,1504.688599,5.739348e-02,20001_10001,NaN
644051,20001,2019-12-31,10002,0.0,20,115.303223,113.331650,69.755478,HC,ROPA LAVADO,...,0.049140,12774,0.001566,2984.121826,26217.066406,0.113824,1504.688599,5.739348e-02,20001_10002,NaN
644052,20001,2019-12-31,10003,0.0,9,113.981369,102.275169,69.755478,HC,ROPA LAVADO,...,0.022113,4013,0.002243,1032.901245,26217.066406,0.039398,1504.688599,5.739348e-02,20001_10003,NaN
644053,20001,2019-12-31,10004,0.0,8,34.648102,34.648102,69.755478,HC,ROPA LAVADO,...,0.019656,2238,0.003575,672.884216,26217.066406,0.025666,1504.688599,5.739348e-02,20001_10004,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664540,21276,2019-12-31,10016,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,0.000000,4393,0.000000,445.574280,26217.066406,0.016996,0.008920,3.402364e-07,21276_10016,NaN
664541,21276,2019-12-31,10017,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,0.000000,566,0.000000,195.421356,26217.066406,0.007454,0.008920,3.402364e-07,21276_10017,NaN
664542,21276,2019-12-31,10018,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,0.000000,1568,0.000000,254.466873,26217.066406,0.009706,0.008920,3.402364e-07,21276_10018,NaN
664543,21276,2019-12-31,10019,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,0.000000,237,0.000000,564.693176,26217.066406,0.021539,0.008920,3.402364e-07,21276_10019,NaN


In [ ]:
test_df["serie_id"] = test_df["product_id"].astype(str) + "_" + test_df["customer_id"].astype(str)
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions[["product_id", "customer_id"]] = predictions['item_id'].str.split('_', expand=True)

predictions = predictions.rename(columns={'item_id': 'serie_id'})
predictions = predictions.merge(test_df[['serie_id', 'target']].drop_duplicates(), on='serie_id', how='left')
# make product_id as int16
predictions["product_id"] = predictions["product_id"].astype("int16")
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
print(predictions)
submission = predictions.reset_index()[['product_id', 'mean']].rename(columns={'mean': 'tn'})
submission['tn'] = submission['tn'].clip(lower=0)  # Asegurar que las predicciones no sean negativas
submission.to_csv("submission_autogluon.csv", index=False)
submission

Total Absolute Error: 2350475431679.7051
                   mean  target    abs_error
product_id                                  
20001       1073.785679     0.0  1073.785679
20002        839.542455     0.0   839.542455
20003        664.910683     0.0   664.910683
20004        507.223147     0.0   507.223147
20005        483.899054     0.0   483.899054
...                 ...     ...          ...
21263          0.010220     0.0     0.010220
21265          0.038337     0.0     0.038337
21266          0.045710     0.0     0.045710
21267          0.055159     0.0     0.055159
21276          0.021705     0.0     0.021705

[780 rows x 3 columns]


,product_id,tn
0,20001,1073.785679
1,20002,839.542455
2,20003,664.910683
3,20004,507.223147
4,20005,483.899054
...,...,...
775,21263,0.010220
776,21265,0.038337
777,21266,0.045710
778,21267,0.055159


In [ ]:
predictor.fit(
    train_data,
    presets="medium_quality"
)
test_pred = predictor.predict(train_data, known_covariates=future_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(test_df["product_id"])]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")

Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_224117'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.58 GB / 15.32 GB (36.4%)
Disk Space Avail:   106.38 GB / 575.67 GB (18.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': ['year', 'mes', 'quarter', 'date_id'],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequ

Total Absolute Error: 0.3307


In [ ]:
base_features = ['product_id',
 'fecha',
 'cust_request_qty',
 'cust_request_qty_agg_std',
 'cust_request_qty_agg_mean',
 'cust_request_qty_agg_max',
 'cust_request_tn',
 'tn',
 'tn_agg_std',
 'tn_agg_mean',
 'tn_agg_max',
 'stock_final',
 'customer_id_tn_zero_count',
'year',
'mes',
'quarter',
'date_id',
]

known_covariate_names = [
    'year',
    'mes',
    'quarter',
    'date_id',
]



static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
static_features_df.isna().sum()
train_df_no_static = train_df_no_static[base_features]
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static, 
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df,
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    known_covariates_names=known_covariate_names

)

# al dataset futuro le agrego las columnas de covariables conocidas
future_data = predictor.make_future_data_frame(train_data)
future_data["year"] = future_data["timestamp"].dt.year
future_data["mes"] = future_data["timestamp"].dt.month
future_data["quarter"] = future_data["timestamp"].dt.quarter
# date id es el last_date id de train +1 y +2 para cada fecha
last_date_id = test_df['date_id'].max()
# creo un date_id para el futuro (1 para la primer fecha 2 para la segunda) y le sumo el ultimo date_id de test_df
num_dates = future_data['timestamp'].unique()
future_data["date_id"] =0
for i, date in enumerate(num_dates):
    # le sumo el ultimo date_id de test_df
    future_data.loc[future_data['timestamp'] == date, 'date_id'] = last_date_id + i + 1


predictor.fit(
    train_data,
    hyperparameters={
    "Chronos": [
        # Zero-shot model WITHOUT covariates
        {
            "model_path": "bolt_small",
            "ag_args": {"name_suffix": "ZeroShot"},
        },
        # Chronos-Bolt (Small) combined with CatBoost on covariates
        {
            "model_path": "bolt_small",
            "covariate_regressor": "CAT",
            "target_scaler": "standard",
            "ag_args": {"name_suffix": "WithRegressor"},
        },
    ],
},
)
test_pred = predictor.predict(train_data, known_covariates=future_data)
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["mean"] = predictions["mean"].clip(lower=0)  # Asegurar que las predicciones no sean negativas

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
predictions

No path specified. Models will be saved in: "AutogluonModels/ag-20250627_235507"
data with frequency 'IRREG' has been resampled to frequency 'ME'.
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250627_235507'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       5.03 GB / 15.32 GB (32.8%)
Disk Space Avail:   106.19 GB / 575.67 GB (18.4%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': {'Chronos': [{'ag_args': {'name_suffix': 'ZeroShot'},
                                  'model_path': 'bolt_small'},
                                 {'ag_args': {'name_suffix': 'WithRegressor'},
                                  'covaria

Total Absolute Error: 0.2553


,product_id,mean,target,abs_error
0,20001,1377.109759,1504.688599,127.578840
1,20002,1149.704390,1087.308594,62.395796
2,20003,862.061270,892.501282,30.440012
3,20004,650.189358,637.900024,12.289334
4,20005,697.208514,593.244446,103.964068
...,...,...,...,...
1193,21263,0.000000,0.012700,0.012700
1195,21265,0.058793,0.050070,0.008723
1196,21266,0.063153,0.051210,0.011943
1197,21267,0.065576,0.015690,0.049886


In [ ]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index

if not "serie_id" in df:
    df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
    
test_index_pred_final, train_index_pred_final, _ = get_indexes(df, 35)
train_df = df.loc[train_index_pred_final].copy()
test_df = df.loc[test_index_pred_final].copy()

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"DF shape: {df.shape}")


static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])

agg_dict = {}
for col in train_df_no_static.columns:
    if col in ['product_id', 'fecha']:
        continue
    elif col == 'tn':
        agg_dict[col] = 'sum'
    elif train_df_no_static[col].dtype == 'O':
        agg_dict[col] = 'first'
    else:
        agg_dict[col] = 'mean'

static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_no_static.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()

from autogluon.timeseries.metrics import TimeSeriesScorer
from autogluon.common.utils.log_utils import set_logger_verbosity

set_logger_verbosity(4)  # Máxima verbosidad (debug)


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
)


predictor.fit(
    train_data,
    #presets="medium_quality",
    #time_limit=600,
)
test_pred = predictor.predict(train_data)
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id'})

# le agrego la columna target de test_df mergeando por product_id
predictions = predictions.merge(test_df[['product_id', 'target']].drop_duplicates(), on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]

predictions = predictions.groupby("product_id").agg({
    "mean": "sum",
    "target": "sum"
})

predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum() + 1e-8)
print(f"Total Absolute Error: {total_error:.4f}")
submission = predictions[['product_id', 'mean']].rename(columns={'mean': 'tn'})
submission['tn'] = submission['tn'].clip(lower=0)  # Asegurar que las predicciones no sean negativas
submission.to_csv("submission_autogluon_basico.csv", index=False)
submission

Train shape: (664545, 119), Test shape: (20496, 119)
DF shape: (664545, 119)


No path specified. Models will be saved in: "AutogluonModels/ag-20250630_030016"
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250630_030016'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       7.06 GB / 15.32 GB (46.1%)
Disk Space Avail:   76.88 GB / 575.67 GB (13.4%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_dat

Total Absolute Error: 125971792141.0508


KeyError: "['product_id'] not in index"

In [ ]:
predictions

,mean,target,abs_error
product_id,,,
20001,62.902541,0.0,62.902541
20002,51.160956,0.0,51.160956
20003,31.172946,0.0,31.172946
20004,22.635965,0.0,22.635965
20005,21.960406,0.0,21.960406
...,...,...,...
21263,0.000175,0.0,0.000175
21265,0.003090,0.0,0.003090
21266,0.003187,0.0,0.003187
